In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_posting_skill = spark.table("job_skills.silver_posting_skill")

display(silver_posting_skill.limit(10))

posting_id,company_id,posted_date,skill_raw,skill
JP040516,CMP116,2025-02-01,Kafka,kafka
JP040516,CMP116,2025-02-01,PowerBI,powerbi
JP040516,CMP116,2025-02-01,Docker,docker
JP040516,CMP116,2025-02-01,Databricks,databricks
JP040546,CMP146,2025-02-03,Spark,spark
JP040546,CMP146,2025-02-03,AWS,aws
JP040546,CMP146,2025-02-03,PowerBI,powerbi
JP040546,CMP146,2025-02-03,Docker,docker
JP040546,CMP146,2025-02-03,Databricks,databricks
JP040579,CMP179,2025-02-08,Airflow,airflow


In [0]:
print("Rows:", silver_posting_skill.count())
print("Unique postings:", silver_posting_skill.select("posting_id").distinct().count())
print("Unique skills:", silver_posting_skill.select("skill").distinct().count())

Rows: 227624
Unique postings: 51840
Unique skills: 20


In [0]:
silver = silver_posting_skill.withColumn(
    "posted_date",
    F.to_date("posted_date")
)

In [0]:
silver = silver.withColumn(
    "month",
    F.date_format("posted_date", "yyyy-MM")
)

In [0]:
display(
    silver.select(
        "posting_id",
        "posted_date",
        "month",
        "skill"
    ).limit(20)
)

posting_id,posted_date,month,skill
JP040516,2025-02-01,2025-02,kafka
JP040516,2025-02-01,2025-02,powerbi
JP040516,2025-02-01,2025-02,docker
JP040516,2025-02-01,2025-02,databricks
JP040546,2025-02-03,2025-02,spark
JP040546,2025-02-03,2025-02,aws
JP040546,2025-02-03,2025-02,powerbi
JP040546,2025-02-03,2025-02,docker
JP040546,2025-02-03,2025-02,databricks
JP040579,2025-02-08,2025-02,airflow


In [0]:
skill_month = (
    silver
    .groupBy("skill", "month")
    .agg(
        F.countDistinct("posting_id").alias("posting_count")
    )
)

In [0]:
display(
    skill_month.orderBy(
        "month",
        F.desc("posting_count")
    )
)

skill,month,posting_count
python,2024-01,2880
hadoop,2024-01,860
azure,2024-01,586
aws,2024-01,586
kafka,2024-01,586
hive,2024-01,586
sql,2024-01,585
airflow,2024-01,585
spark,2024-01,585
gcp,2024-01,581


In [0]:
monthly_total = (
    silver
    .groupBy("month")
    .agg(
        F.countDistinct("posting_id").alias("postings_that_month")
    )
)

In [0]:
display(monthly_total.orderBy("month"))

month,postings_that_month
2024-01,2880
2024-02,2880
2024-03,2880
2024-04,2880
2024-05,2880
2024-06,2880
2024-07,2880
2024-08,2880
2024-09,2880
2024-10,2880


In [0]:
print("Months:", monthly_total.count())

Months: 18


In [0]:
gold_skill_month = (
    skill_month
    .join(
        monthly_total,
        on="month",
        how="left"
    )
    .withColumn(
        "skill_share",
        F.round(
            F.col("posting_count") /
            F.col("postings_that_month"),
            4
        )
    )
)

In [0]:
display(
    gold_skill_month.orderBy(
        "month",
        F.desc("posting_count")
    )
)

month,skill,posting_count,postings_that_month,skill_share
2024-01,python,2880,2880,1.0
2024-01,hadoop,860,2880,0.2986
2024-01,hive,586,2880,0.2035
2024-01,aws,586,2880,0.2035
2024-01,azure,586,2880,0.2035
2024-01,kafka,586,2880,0.2035
2024-01,airflow,585,2880,0.2031
2024-01,sql,585,2880,0.2031
2024-01,spark,585,2880,0.2031
2024-01,gcp,581,2880,0.2017


In [0]:
print("Gold skill-month rows:", gold_skill_month.count())
print(
    "Unique skills:",
    gold_skill_month.select("skill").distinct().count()
)
print(
    "Unique months:",
    gold_skill_month.select("month").distinct().count()
)

Gold skill-month rows: 360
Unique skills: 20
Unique months: 18


In [0]:
print("Gold skill-month rows:", gold_skill_month.count())
print("Unique skills:", gold_skill_month.select("skill").distinct().count())
print("Unique months:", gold_skill_month.select("month").distinct().count())

Gold skill-month rows: 360
Unique skills: 20
Unique months: 18


In [0]:
gold_skill_month.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/job_skills/raw_files/raw/skill_month")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS job_skills.gold_skill_month
USING DELTA
as select * from DELTA .`/Volumes/workspace/job_skills/raw_files/raw/skill_month`
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SHOW TABLES IN job_skills").show()

+----------+--------------------+-----------+
|  database|           tableName|isTemporary|
+----------+--------------------+-----------+
|job_skills|     bronze_postings|      false|
|job_skills|    gold_skill_month|      false|
|job_skills|gold_skill_month_...|      false|
|job_skills|    gold_skill_pairs|      false|
|job_skills|gold_skill_quarte...|      false|
|job_skills|silver_posting_skill|      false|
|job_skills|silver_rejected_p...|      false|
|job_skills|silver_valid_post...|      false|
+----------+--------------------+-----------+



What are the top 10 skills by posting count in each month?

In [0]:
window_month = (
    Window
    .partitionBy("month")
    .orderBy(F.desc("posting_count"))
)

In [0]:
top10 = (
    gold_skill_month
    .withColumn(
        "rank",
        F.row_number().over(window_month)
    )
    .filter(
        F.col("rank") <= 10
    )
)

In [0]:
display(
    top10.orderBy(
        "month",
        "rank"
    )
)

month,skill,posting_count,postings_that_month,skill_share,rank
2024-01,python,2880,2880,1.0,1
2024-01,hadoop,860,2880,0.2986,2
2024-01,aws,586,2880,0.2035,3
2024-01,hive,586,2880,0.2035,4
2024-01,azure,586,2880,0.2035,5
2024-01,kafka,586,2880,0.2035,6
2024-01,sql,585,2880,0.2031,7
2024-01,spark,585,2880,0.2031,8
2024-01,airflow,585,2880,0.2031,9
2024-01,gcp,581,2880,0.2017,10


In [0]:
print("Top 10 rows:", top10.count())
print("Months:", top10.select("month").distinct().count())

Top 10 rows: 180
Months: 18


In [0]:
display(
    top10
    .groupBy("month")
    .count()
    .orderBy("month")
)

month,count
2024-01,10
2024-02,10
2024-03,10
2024-04,10
2024-05,10
2024-06,10
2024-07,10
2024-08,10
2024-09,10
2024-10,10


In [0]:
top10.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/job_skills/raw_files/raw/skill_month_top10")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS job_skills.gold_skill_month_top10
USING DELTA as select * from delta .`/Volumes/workspace/job_skills/raw_files/raw/skill_month_top10`
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql 
show tables in job_skills;

database,tableName,isTemporary
job_skills,bronze_postings,false
job_skills,gold_skill_month,false
job_skills,gold_skill_month_top10,false
job_skills,gold_skill_pairs,false
job_skills,gold_skill_quarter_growth,false
job_skills,silver_posting_skill,false
job_skills,silver_rejected_postings,false
job_skills,silver_valid_postings,false


In [0]:
quarter_data = (
    gold_skill_month
    .withColumn(
        "date",
        F.to_date(F.concat(F.col("month"), F.lit("-01")))
    )
    .withColumn(
        "quarter",
        F.concat(
            F.year("date"),
            F.lit("-Q"),
            F.quarter("date")
        )
    )
)

In [0]:
display(
    quarter_data.select(
        "skill",
        "month",
        "quarter",
        "posting_count",
        "skill_share"
    ).orderBy("month").limit(20)
)

skill,month,quarter,posting_count,skill_share
kubernetes,2024-01,2024-Q1,476,0.1653
hadoop,2024-01,2024-Q1,860,0.2986
docker,2024-01,2024-Q1,570,0.1979
hive,2024-01,2024-Q1,586,0.2035
azure,2024-01,2024-Q1,586,0.2035
java,2024-01,2024-Q1,329,0.1142
sql,2024-01,2024-Q1,585,0.2031
python,2024-01,2024-Q1,2880,1.0
dbt,2024-01,2024-Q1,260,0.0903
powerbi,2024-01,2024-Q1,329,0.1142


In [0]:
quarter_skill = (
    quarter_data
    .groupBy("skill", "quarter")
    .agg(
        F.sum("posting_count").alias("quarter_postings")
    )
)

In [0]:
quarter_total = (
    quarter_data
    .groupBy("quarter")
    .agg(
        F.sum("postings_that_month").alias("quarter_total_postings")
    )
    .dropDuplicates(["quarter"])
)

In [0]:
quarter_skill = (
    quarter_skill
    .join(
        quarter_total,
        on="quarter",
        how="left"
    )
    .withColumn(
        "quarter_share",
        F.round(
            F.col("quarter_postings") /
            F.col("quarter_total_postings"),
            4
        )
    )
)

In [0]:
display(
    quarter_skill.orderBy(
        "quarter",
        F.desc("quarter_share")
    )
)

quarter,skill,quarter_postings,quarter_total_postings,quarter_share
2024-Q1,python,5003,172800,0.029
2024-Q1,sql,4998,172800,0.0289
2024-Q1,hadoop,2466,172800,0.0143
2024-Q1,spark,2115,172800,0.0122
2024-Q1,snowflake,1626,172800,0.0094
2024-Q1,airflow,1537,172800,0.0089
2024-Q1,scala,1499,172800,0.0087
2024-Q1,powerbi,1499,172800,0.0087
2024-Q1,gcp,1496,172800,0.0087
2024-Q1,tableau,1496,172800,0.0087


In [0]:
quarter_window = (
    Window
    .partitionBy("skill")
    .orderBy("quarter")
)

In [0]:
quarter_growth = (
    quarter_skill
    .withColumn(
        "previous_quarter_share",
        F.lag("quarter_share").over(quarter_window)
    )
    .withColumn(
        "qoq_change",
        F.round(
            F.col("quarter_share") -
            F.col("previous_quarter_share"),
            4
        )
    )
)

In [0]:
display(
    quarter_growth.orderBy(
        "skill",
        "quarter"
    )
)

quarter,skill,quarter_postings,quarter_total_postings,quarter_share,previous_quarter_share,qoq_change
2024-Q1,airflow,1537,172800,0.0089,null,null
2024-Q2,airflow,4966,172800,0.0287,0.0089,0.0198
2024-Q3,airflow,1500,172800,0.0087,0.0287,-0.02
2024-Q4,airflow,1497,172800,0.0087,0.0087,0.0
2025-Q1,airflow,1414,172800,0.0082,0.0087,-5.0E-4
2025-Q2,airflow,1489,172800,0.0086,0.0082,4.0E-4
2024-Q1,aws,1389,172800,0.008,null,null
2024-Q2,aws,1570,172800,0.0091,0.008,0.0011
2024-Q3,aws,3465,172800,0.0201,0.0091,0.011
2024-Q4,aws,2977,172800,0.0172,0.0201,-0.0029


In [0]:
quarter_growth.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/job_skills/raw_files/raw/skill_quarter_growth")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS job_skills.gold_skill_quarter_growth
USING DELTA
as select * from delta .`/Volumes/workspace/job_skills/raw_files/raw/skill_quarter_growth`
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Rows:", quarter_growth.count())

Rows: 120


In [0]:
a = (
    silver
    .select("posting_id", "month", "skill")
    .distinct()
    .withColumnRenamed("skill", "skill_a")
)

b = (
    silver
    .select("posting_id", "month", "skill")
    .distinct()
    .withColumnRenamed("skill", "skill_b")
)

pairs = (
    a.join(
        b,
        on=["posting_id", "month"],
        how="inner"
    )
    .filter(F.col("skill_a") != F.col("skill_b"))
)

In [0]:
pair_counts = (
    pairs
    .groupBy(
        "skill_a",
        "skill_b",
        "month"
    )
    .agg(
        F.countDistinct("posting_id").alias("pair_postings")
    )
)

In [0]:
display(
    pair_counts.orderBy(
        "month",
        F.desc("pair_postings")
    )
)

skill_a,skill_b,month,pair_postings
python,hadoop,2024-01,860
hadoop,python,2024-01,860
aws,python,2024-01,586
hive,python,2024-01,586
python,kafka,2024-01,586
azure,python,2024-01,586
python,azure,2024-01,586
python,hive,2024-01,586
python,aws,2024-01,586
kafka,python,2024-01,586


In [0]:
skill_posting_counts = (
    silver
    .groupBy(
        "skill",
        "month"
    )
    .agg(
        F.countDistinct("posting_id").alias("skill_postings")
    )
)

In [0]:
skill_a_counts = skill_posting_counts.select(
    F.col("skill").alias("skill_a"),
    "month",
    F.col("skill_postings").alias("skill_a_postings")
)

In [0]:
skills = silver.select("skill").distinct()

In [0]:
all_pairs = (
    skills.alias("a")
    .crossJoin(
        skills.alias("b")
    )
    .filter(
        F.col("a.skill") != F.col("b.skill")
    )
    .select(
        F.col("a.skill").alias("skill_a"),
        F.col("b.skill").alias("skill_b")
    )
)

In [0]:
print("Possible skill pairs:", all_pairs.count())

Possible skill pairs: 380


In [0]:
months = silver.select("month").distinct()

In [0]:
all_pair_months = (
    all_pairs
    .crossJoin(months)
)

In [0]:
print("Possible pair-month rows:", all_pair_months.count())

Possible pair-month rows: 6840


In [0]:
complete_pairs = (
    all_pair_months
    .join(
        pair_counts,
        on=[
            "skill_a",
            "skill_b",
            "month"
        ],
        how="left"
    )
    .fillna(
        {"pair_postings": 0}
    )
)

In [0]:
complete_pairs = (
    complete_pairs
    .join(
        skill_a_counts,
        on=[
            "skill_a",
            "month"
        ],
        how="left"
    )
)

In [0]:
complete_pairs = complete_pairs.withColumn(
    "confidence_a_to_b",
    F.when(
        F.col("skill_a_postings") > 0,
        F.round(
            F.col("pair_postings") /
            F.col("skill_a_postings"),
            4
        )
    ).otherwise(0)
)
overall_pairs = (
    pairs
    .groupBy("skill_a", "skill_b")
    .agg(
        F.countDistinct("posting_id").alias("companion_postings")
    )
)

skill_totals = (
    silver
    .select("posting_id", "skill")
    .distinct()
    .groupBy("skill")
    .agg(
        F.countDistinct("posting_id").alias("skill_postings")
    )
    .withColumnRenamed("skill", "skill_a")
)

overall_companions = (
    overall_pairs
    .join(skill_totals, on="skill_a", how="left")
    .withColumn(
        "confidence_a_to_b",
        F.round(
            F.col("companion_postings") /
            F.col("skill_postings"),
            4
        )
    )
)

In [0]:
display(
    overall_companions
    .filter(F.col("skill_a") == "snowflake")
    .orderBy(F.desc("confidence_a_to_b"))
)

skill_a,skill_b,companion_postings,skill_postings,confidence_a_to_b
snowflake,dbt,6363,12726,0.5
snowflake,gcp,3709,12726,0.2915
snowflake,terraform,3615,12726,0.2841
snowflake,scala,3160,12726,0.2483
snowflake,hive,3132,12726,0.2461
snowflake,python,3045,12726,0.2393
snowflake,kubernetes,3031,12726,0.2382
snowflake,airflow,3020,12726,0.2373
snowflake,spark,2967,12726,0.2331
snowflake,aws,2883,12726,0.2265


In [0]:
gold_skill_pairs = complete_pairs.select(
    "skill_a",
    "skill_b",
    "month",
    "pair_postings",
    "confidence_a_to_b"
)

In [0]:
display(
    gold_skill_pairs
    .filter(F.col("skill_a") == "snowflake")
    .orderBy(
        "month",
        F.desc("confidence_a_to_b")
    )
)

skill_a,skill_b,month,pair_postings,confidence_a_to_b
snowflake,python,2024-01,520,1.0
snowflake,sql,2024-01,288,0.5538
snowflake,dbt,2024-01,260,0.5
snowflake,spark,2024-01,70,0.1346
snowflake,hive,2024-01,63,0.1212
snowflake,azure,2024-01,63,0.1212
snowflake,kafka,2024-01,63,0.1212
snowflake,aws,2024-01,63,0.1212
snowflake,airflow,2024-01,63,0.1212
snowflake,terraform,2024-01,62,0.1192


In [0]:
print(
    "Total pair-month rows:",
    gold_skill_pairs.count()
)

Total pair-month rows: 6840


In [0]:
print(
    "Non-zero pairs:",
    gold_skill_pairs
    .filter(F.col("pair_postings") > 0)
    .count()
)

Non-zero pairs: 6660


In [0]:
print(
    "Zero pairs:",
    gold_skill_pairs
    .filter(F.col("pair_postings") == 0)
    .count()
)

Zero pairs: 180


In [0]:
display(
    gold_skill_pairs
    .filter(
        F.col("skill_a") == "snowflake"
    )
    .filter(
        F.col("pair_postings") > 0
    )
    .orderBy(
        F.desc("confidence_a_to_b")
    )
)

skill_a,skill_b,month,pair_postings,confidence_a_to_b
snowflake,terraform,2025-05,872,1.0
snowflake,kafka,2024-07,652,1.0
snowflake,python,2024-02,542,1.0
snowflake,kubernetes,2025-03,828,1.0
snowflake,spark,2024-04,586,1.0
snowflake,python,2024-01,520,1.0
snowflake,gcp,2024-12,762,1.0
snowflake,gcp,2025-01,784,1.0
snowflake,hive,2024-08,674,1.0
snowflake,aws,2024-10,718,1.0


In [0]:
print(
    gold_skill_pairs
    .filter(
        (F.col("skill_a") == "snowflake") &
        (F.col("skill_b") != "dbt") &
        (F.col("pair_postings") > 0) &
        (F.col("confidence_a_to_b") >= 0.30)
    )
    .count()
)

63


In [0]:
display(
    gold_skill_pairs
    .filter(
        (F.col("skill_a") == "snowflake") &
        (F.col("skill_b") != "dbt") &
        (F.col("pair_postings") > 0) &
        (F.col("confidence_a_to_b") >= 0.30)
    )
    .orderBy(
        F.desc("confidence_a_to_b"),
        "skill_b",
        "month"
    )
)

skill_a,skill_b,month,pair_postings,confidence_a_to_b
snowflake,airflow,2024-06,630,1.0
snowflake,aws,2024-10,718,1.0
snowflake,azure,2024-11,740,1.0
snowflake,docker,2025-02,806,1.0
snowflake,gcp,2024-12,762,1.0
snowflake,gcp,2025-01,784,1.0
snowflake,hive,2024-08,674,1.0
snowflake,kafka,2024-07,652,1.0
snowflake,kubernetes,2025-03,828,1.0
snowflake,python,2024-01,520,1.0


In [0]:
print("Total:", gold_skill_pairs.count())

print(
    "Non-zero:",
    gold_skill_pairs
    .filter(F.col("pair_postings") > 0)
    .count()
)

print(
    "Zero:",
    gold_skill_pairs
    .filter(F.col("pair_postings") == 0)
    .count()
)

print(
    "Null:",
    gold_skill_pairs
    .filter(F.col("pair_postings").isNull())
    .count()
)

Total: 6840
Non-zero: 6660
Zero: 180
Null: 0


In [0]:
gold_skill_pairs.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/job_skills/raw_files/raw/skill_pairs")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS job_skills.gold_skill_pairs
USING DELTA
AS SELECT * FROM DELTA .`/Volumes/workspace/job_skills/raw_files/raw/skill_pairs`
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.table("job_skills.gold_skill_pairs").count()

6840

In [0]:
display(
    gold_skill_pairs
    .filter(F.col("skill_a") == "snowflake")
    .filter(F.col("pair_postings") > 0)
    .orderBy(F.desc("confidence_a_to_b"))
)

skill_a,skill_b,month,pair_postings,confidence_a_to_b
snowflake,terraform,2025-05,872,1.0
snowflake,kafka,2024-07,652,1.0
snowflake,python,2024-02,542,1.0
snowflake,kubernetes,2025-03,828,1.0
snowflake,spark,2024-04,586,1.0
snowflake,python,2024-01,520,1.0
snowflake,gcp,2024-12,762,1.0
snowflake,gcp,2025-01,784,1.0
snowflake,hive,2024-08,674,1.0
snowflake,aws,2024-10,718,1.0


In [0]:
print(
    gold_skill_pairs
    .filter(
        (F.col("skill_a") == "snowflake") &
        (F.col("skill_b") != "dbt") &
        (F.col("pair_postings") > 0) &
        (F.col("confidence_a_to_b") >= 0.30)
    )
    .count()
)

63


In [0]:
skill_per_posting = (
    silver
    .groupBy("posting_id")
    .agg(F.countDistinct("skill").alias("skill_count"))
)

display(
    skill_per_posting
    .groupBy("skill_count")
    .count()
    .orderBy("skill_count")
)

skill_count,count
1,3
2,430
3,6409
4,21582
5,19290
6,4126


In [0]:
display(
    silver
    .filter(F.col("skill") == "snowflake")
    .select("posting_id", "posted_date", "skill")
    .limit(10)
)

posting_id,posted_date,skill
JP042007,2025-03-08,snowflake
JP042060,2025-03-05,snowflake
JP042066,2025-03-11,snowflake
JP042067,2025-03-12,snowflake
JP042086,2025-03-03,snowflake
JP042087,2025-03-04,snowflake
JP042123,2025-03-12,snowflake
JP042127,2025-03-16,snowflake
JP042140,2025-03-01,snowflake
JP042141,2025-03-02,snowflake


In [0]:
snowflake_ids = (
    silver
    .filter(F.col("skill") == "snowflake")
    .select("posting_id")
    .distinct()
)

display(
    silver
    .join(snowflake_ids, on="posting_id", how="inner")
    .select("posting_id", "skill")
    .orderBy("posting_id", "skill")
    .limit(50)
)

posting_id,skill
JP000000,dbt
JP000000,python
JP000000,snowflake
JP000001,python
JP000001,snowflake
JP000001,sql
JP000002,dbt
JP000002,python
JP000002,snowflake
JP000002,spark


In [0]:
display(
    silver
    .filter(
        (F.col("skill") == "snowflake") &
        (F.col("month") == "2025-01")
    )
    .groupBy("posting_id")
    .agg(
        F.collect_set("skill").alias("skills")
    )
    .orderBy("posting_id")
    .limit(20)
)

posting_id,skills
JP036000,List(snowflake)
JP036001,List(snowflake)
JP036002,List(snowflake)
JP036003,List(snowflake)
JP036004,List(snowflake)
JP036005,List(snowflake)
JP036006,List(snowflake)
JP036007,List(snowflake)
JP036008,List(snowflake)
JP036009,List(snowflake)


In [0]:

silver = (
    spark.table("job_skills.silver_posting_skill")
    .withColumn("posted_date", F.to_date("posted_date"))
    .withColumn("month", F.date_format("posted_date", "yyyy-MM"))
)

In [0]:
a = (
    silver
    .select("posting_id", "month", "skill")
    .withColumnRenamed("skill", "skill_a")
)

b = (
    silver
    .select("posting_id", "month", "skill")
    .withColumnRenamed("skill", "skill_b")
)

pairs = (
    a.join(
        b,
        on=["posting_id", "month"],
        how="inner"
    )
    .filter(F.col("skill_a") != F.col("skill_b"))
)

In [0]:
pair_counts = (
    pairs
    .groupBy("skill_a", "skill_b", "month")
    .agg(
        F.countDistinct("posting_id").alias("pair_postings")
    )
)

In [0]:
display(
    pairs
    .filter(F.col("posting_id") == "JP036000")
    .orderBy("skill_a", "skill_b")
)

posting_id,month,skill_a,skill_b
JP036000,2025-01,dbt,gcp
JP036000,2025-01,dbt,kubernetes
JP036000,2025-01,dbt,python
JP036000,2025-01,dbt,scala
JP036000,2025-01,dbt,snowflake
JP036000,2025-01,gcp,dbt
JP036000,2025-01,gcp,kubernetes
JP036000,2025-01,gcp,python
JP036000,2025-01,gcp,scala
JP036000,2025-01,gcp,snowflake


In [0]:
pair_counts = (
    pairs
    .groupBy("skill_a", "skill_b", "month")
    .agg(
        F.countDistinct("posting_id").alias("pair_postings")
    )
)

In [0]:
print("Total:", gold_skill_pairs.count())

print(
    "Non-zero:",
    gold_skill_pairs
    .filter(F.col("pair_postings") > 0)
    .count()
)

print(
    "Zero:",
    gold_skill_pairs
    .filter(F.col("pair_postings") == 0)
    .count()
)

print(
    "Null:",
    gold_skill_pairs
    .filter(F.col("pair_postings").isNull())
    .count()
)

Total: 6840
Non-zero: 6660
Zero: 180
Null: 0


In [0]:
display(
    gold_skill_pairs
    .filter(
        (F.col("skill_a") == "snowflake") &
        (F.col("pair_postings") > 0)
    )
    .orderBy(F.desc("confidence_a_to_b"))
    .limit(30)
)

skill_a,skill_b,month,pair_postings,confidence_a_to_b
snowflake,gcp,2024-12,762,1.0
snowflake,gcp,2025-01,784,1.0
snowflake,docker,2025-02,806,1.0
snowflake,aws,2024-10,718,1.0
snowflake,python,2024-02,542,1.0
snowflake,spark,2024-04,586,1.0
snowflake,kubernetes,2025-03,828,1.0
snowflake,kafka,2024-07,652,1.0
snowflake,azure,2024-11,740,1.0
snowflake,airflow,2024-06,630,1.0


In [0]:
display(
    gold_skill_pairs
    .filter(F.col("skill_a") == "snowflake")
    .filter(F.col("confidence_a_to_b") >= 0.30)
    .orderBy("month", "skill_b")
)

skill_a,skill_b,month,pair_postings,confidence_a_to_b
snowflake,dbt,2024-01,260,0.5
snowflake,python,2024-01,520,1.0
snowflake,sql,2024-01,288,0.5538
snowflake,dbt,2024-02,271,0.5
snowflake,java,2024-02,265,0.4889
snowflake,python,2024-02,542,1.0
snowflake,scala,2024-02,304,0.5609
snowflake,azure,2024-03,200,0.3546
snowflake,dbt,2024-03,282,0.5
snowflake,docker,2024-03,219,0.3883


In [0]:
from pyspark.sql import functions as F

silver = (
    spark.table("job_skills.silver_posting_skill")
    .withColumn("posted_date", F.to_date("posted_date"))
    .withColumn("month", F.date_format("posted_date", "yyyy-MM"))
)

snowflake_posts = (
    silver
    .filter(F.col("skill") == "snowflake")
    .select("posting_id", "month")
    .distinct()
)

snowflake_companions = (
    snowflake_posts
    .join(
        silver.select("posting_id", "month", "skill").distinct(),
        on=["posting_id", "month"],
        how="inner"
    )
    .filter(F.col("skill") != "snowflake")
)

companion_check = (
    snowflake_companions
    .groupBy("skill")
    .agg(
        F.countDistinct("posting_id").alias("companion_postings")
    )
)

snowflake_total = snowflake_posts.count()

companion_check = companion_check.withColumn(
    "confidence",
    F.round(
        F.col("companion_postings") / F.lit(snowflake_total),
        4
    )
)

display(
    companion_check
    .orderBy(F.desc("confidence"))
)

skill,companion_postings,confidence
dbt,6363,0.5
gcp,3709,0.2915
terraform,3615,0.2841
scala,3160,0.2483
hive,3132,0.2461
python,3045,0.2393
kubernetes,3031,0.2382
airflow,3020,0.2373
spark,2967,0.2331
aws,2883,0.2265


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/job_skills/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/job_skills/raw_files/bronze_postings/,bronze_postings/,0,1790506359438
dbfs:/Volumes/workspace/job_skills/raw_files/raw/,raw/,0,1790506359438
dbfs:/Volumes/workspace/job_skills/raw_files/silver/,silver/,0,1790506359438


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/job_skills/raw_files"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/job_skills/raw_files/bronze_postings/,bronze_postings/,0,1790506481343
dbfs:/Volumes/workspace/job_skills/raw_files/raw/,raw/,0,1790506481343
dbfs:/Volumes/workspace/job_skills/raw_files/silver/,silver/,0,1790506481343


In [0]:
spark.sql("SHOW TABLES IN job_skills").show(truncate=False)

+----------+-------------------------+-----------+
|database  |tableName                |isTemporary|
+----------+-------------------------+-----------+
|job_skills|bronze_postings          |false      |
|job_skills|gold_skill_month         |false      |
|job_skills|gold_skill_month_top10   |false      |
|job_skills|gold_skill_pairs         |false      |
|job_skills|gold_skill_quarter_growth|false      |
|job_skills|silver_posting_skill     |false      |
|job_skills|silver_rejected_postings |false      |
|job_skills|silver_valid_postings    |false      |
+----------+-------------------------+-----------+



In [0]:
for table in [
    "gold_skill_month",
    "gold_skill_month_top10",
    "gold_skill_pairs",
    "gold_skill_quarter_growth"
]:
    print(table, spark.table(f"job_skills.{table}").count())

gold_skill_month 360
gold_skill_month_top10 180
gold_skill_pairs 6840
gold_skill_quarter_growth 120


In [0]:
tables = {
    "skill_month": "job_skills.gold_skill_month",
    "skill_month_top10": "job_skills.gold_skill_month_top10",
    "skill_quarter_growth": "job_skills.gold_skill_quarter_growth",
    "skill_pairs": "job_skills.gold_skill_pairs"
}

for folder, table in tables.items():
    (
        spark.table(table)
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(f"/Volumes/workspace/job_skills/raw_files/gold_csv/{folder}")
    )

print("All 4 Gold tables exported as CSV.")

All 4 Gold tables exported as CSV.


In [0]:
df = spark.table("job_skills.silver_posting_skill")

df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .option("delimiter", ",") \
    .csv("/Volumes/workspace/job_skills/raw_files/gold_csv/silver_posting_skill")